# AI-Powered Resume Screening & Job Matching System


## Data preprocessing, semantic search, skill extraction and candidate ranking.


### Technologies
- Python
- Pandas
- Sentence Transformers
- spaCy
- Scikit-learn

# Dataset Loadin

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


KeyError: '_items'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install faiss-cpu
!pip install flashtext

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

job_df = pd.read_csv('/content/drive/MyDrive/CSVfiles/job_descriptions.csv')
job_df.shape

(1615940, 23)

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"
DATA_PATH = BASE_PATH + "data/"
MODEL_PATH = BASE_PATH + "models/"

In [ ]:
# DATA_PATH = "data/"

# resume_df = pd.read_csv(DATA_PATH + "Resume.csv")
# job_df = pd.read_csv(DATA_PATH + "job_descriptions.csv")

In [ ]:
resume_df = pd.read_csv('/content/drive/MyDrive/CSVfiles/Resume.csv')
resume_df.shape

(2484, 4)

# Data Cleaning

In [ ]:
resume_df.isnull().sum()

,0
ID,0
Resume_str,0
Resume_html,0
Category,0


In [ ]:
job_df.isnull().sum()

,0
Job Id,0
Experience,0
Qualifications,0
Salary Range,0
location,0
Country,0
latitude,0
longitude,0
Work Type,0
Company Size,0


In [ ]:
# job_df1 = job_df.dropna()
# job_df1.shape

In [ ]:
job_df2 = job_df.drop(columns=[
    "Job Id",
    "latitude",
    "longitude",
    "Job Posting Date",
    "Preference",
    "Contact Person",
    "Contact",
    "Job Portal",
    "Company Profile",
    "Company Size"
])
len(job_df2.columns)

13

In [ ]:
job_df2["job_text"] = (
    job_df2["Job Title"] + " " +
    job_df2["Role"] + " " +
    job_df2["Job Description"] + " " +
    job_df2["skills"] + " " +
    job_df2["Responsibilities"] + " " +
    job_df2["Qualifications"] + " " +
    job_df2["Experience"]
)

In [ ]:
resume_df["resume_text"] = resume_df["Resume_str"]

In [ ]:
import re
import html
import unicodedata


def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = html.unescape(text)

    text = unicodedata.normalize("NFKC", text)

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"www\S+", " ", text)

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"[^a-zA-Z0-9+#./ ]", " ", text)

    text = text.lower()

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
resume_df["raw_resume"] = resume_df["resume_text"]
resume_df["resume_text"] = resume_df["resume_text"].apply(clean_text)
resume_df.head()

,ID,Resume_str,Resume_html,Category,resume_text,raw_resume
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr administrator/marketing associate hr admini...,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist us hr operations summary versati...,"HR SPECIALIST, US HR OPERATIONS ..."
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr director summary over 20 years experience i...,HR DIRECTOR Summary Over 2...
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist summary dedicated driven and dyn...,HR SPECIALIST Summary Dedica...
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr manager skill highlights hr skills hr depar...,HR MANAGER Skill Highlights ...


In [ ]:
job_df2["raw_job"] = job_df2["job_text"]
job_df2["job_text"] = job_df2["job_text"].apply(clean_text)
job_df2.head()

,Experience,Qualifications,Salary Range,location,Country,Work Type,Job Title,Role,Job Description,Benefits,skills,Responsibilities,Company,job_text,raw_job
0,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,Intern,Digital Marketing Specialist,Social Media Manager,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,digital marketing specialist social media mana...,Digital Marketing Specialist Social Media Mana...
1,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,Intern,Web Developer,Frontend Web Developer,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,web developer frontend web developer frontend ...,Web Developer Frontend Web Developer Frontend ...
2,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",Temporary,Operations Manager,Quality Control Manager,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,operations manager quality control manager qua...,Operations Manager Quality Control Manager Qua...
3,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,Full-Time,Network Engineer,Wireless Network Engineer,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,network engineer wireless network engineer wir...,Network Engineer Wireless Network Engineer Wir...
4,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,Intern,Event Manager,Conference Manager,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,event manager conference manager a conference ...,Event Manager Conference Manager A Conference ...


In [ ]:
job_df2['job_text'][0]

'digital marketing specialist social media manager social media managers oversee an organizations social media presence. they create and schedule content engage with followers and analyze social media metrics to drive brand awareness and engagement. social media platforms e.g. facebook twitter instagram content creation and scheduling social media analytics and insights community engagement paid social advertising manage and grow social media accounts create engaging content and interact with the online community. develop social media content calendars and strategies. monitor social media trends and engagement metrics. m.tech 5 to 15 years'

In [ ]:
resume_df["Category"].value_counts()

,count
Category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,120
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,118
FITNESS,117
AVIATION,117


In [ ]:
job_df["Job Title"].nunique()

147

# Embedding Generation


In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
resume_embeddings = model.encode(
    resume_df["resume_text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
resume_embeddings.shape

(2484, 384)

In [ ]:
import numpy as np
import os


texts = job_df2["job_text"].tolist()

chunk_size = 100000

MODEL_PATH = "/content/drive/MyDrive/AI Resume Screening/models/"
os.makedirs(MODEL_PATH, exist_ok=True)

for i, start in enumerate(range(0, len(texts), chunk_size)):

    end = min(start + chunk_size, len(texts))

    emb = model.encode(
        texts[start:end],
        batch_size=512,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    np.save(
        MODEL_PATH + f"job_embeddings_part_{i}.npy",
        emb
    )


In [ ]:
import glob
import faiss
import numpy as np
import os

files = sorted(glob.glob(os.path.join(MODEL_PATH, "job_embeddings_part_*.npy")))

first = np.load(files[0])
dimension = first.shape[1]

index = faiss.IndexFlatIP(dimension)

for file in files:
    emb = np.load(file)
    index.add(emb)

faiss.write_index(index, MODEL_PATH + "job_faiss.index")

print(index.ntotal)

1615940


In [ ]:
with open(MODEL_PATH + "embedding_model.txt", "w") as f:
    f.write(MODEL_NAME)

# Job Recommendation

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def recommend_jobs(resume_idx, top_n=5):
    if resume_idx >= len(resume_df):
      raise ValueError("Invalid resume index")

    resume_embedding = resume_embeddings[resume_idx].reshape(1, -1)

    scores, indices = index.search(resume_embedding, top_n)
    top_jobs = scores[0].argsort()[-top_n:][::-1]

    recommendations = []

    resume_text = resume_df.iloc[resume_idx]["resume_text"].lower()

    for idx in top_jobs:

        skill_text = str(job_df2.iloc[idx]["skills"])

        recommendations.append({
            "Job Title": job_df2.iloc[idx]["Job Title"],
            "Role": job_df2.iloc[idx]["Role"],
            "Match Score": round(scores[0][idx] * 100, 2),
            "Skills Required": skill_text
        })

    return pd.DataFrame(recommendations)

# Skill Extraction

In [ ]:
job_df2["skills"].head()

,skills
0,"Social media platforms (e.g., Facebook, Twitte..."
1,"HTML, CSS, JavaScript Frontend frameworks (e.g..."
2,Quality control processes and methodologies St...
3,Wireless network design and architecture Wi-Fi...
4,Event planning Conference logistics Budget man...


In [ ]:
import re

def extract_skills(skill_text):
    skills = re.split(r'[,;]', str(skill_text))
    return {
        skill.strip().lower()
        for skill in skills
        if len(skill.strip()) > 2
    }

In [ ]:
all_skills = set()

for skills in job_df2["skills"].dropna():
    for skill in str(skills).split(","):
        skill = skill.strip().lower()
        if skill:
            all_skills.add(skill)

In [ ]:
from flashtext import KeywordProcessor

keyword_processor = KeywordProcessor(case_sensitive=False)

for skill in all_skills:
    keyword_processor.add_keyword(skill)

In [ ]:
import os

print(os.path.exists(MODEL_PATH + "job_faiss.index"))
print(index.ntotal)

True
1615940


# Candidate Ranking

In [ ]:
# def rank_candidates(job_idx, top_k=10):

#     # Get embedding of selected job
#     job_embedding = job_embeddings[job_idx].reshape(1,-1)

#     # Similarity with every resume
#     scores = cosine_similarity(job_embedding, resume_embeddings)[0]

#     # Top candidates
#     top_indices = np.argsort(scores)[::-1][:top_k]

#     ranked_candidates = []

#     for idx in top_indices:

#         ranked_candidates.append({

#             "Resume Index": idx,

#             "Similarity Score": round(scores[idx]*100,2),

#             "Category": resume_df.iloc[idx]["Category"],

#             "Resume Text": resume_df.iloc[idx]["resume_text"][:250]

#         })

#     return ranked_candidates

In [ ]:
# rank_candidates(job_idx=15)

In [ ]:
# def rank_candidates_df(job_idx, top_k=10):
#     if job_idx >= len(job_df2):
#       raise ValueError("Invalid resume index")

#     job_embedding = job_embeddings[job_idx].reshape(1,-1)

#     scores = cosine_similarity(job_embedding,resume_embeddings)[0]

#     top_indices = np.argsort(scores)[::-1][:top_k]

#     result = resume_df.iloc[top_indices][
#         ["Category","resume_text"]
#     ].copy()

#     # result["Match Score"] = scores[top_indices]*100

#     result["Match Score"] = result["Match Score"].round(2)

#     return result.reset_index(drop=True)

In [ ]:
def show_job(job_idx):

    print(job_df2.iloc[job_idx]["Job Title"])
    print(job_df2.iloc[job_idx]["Role"])
    print()
    print(job_df2.iloc[job_idx]["Job Description"])

In [ ]:
show_job(15)

Brand Manager
Product Brand Manager

A Product Brand Manager focuses on promoting and managing the brand identity of specific products or product lines within a companys portfolio.


In [ ]:
job_idx = 15

show_job(job_idx)

Brand Manager
Product Brand Manager

A Product Brand Manager focuses on promoting and managing the brand identity of specific products or product lines within a companys portfolio.


# Save Artifacts

In [ ]:
print("=" * 70)

print("Notebook 1 Completed Successfully")

print()

print("Artifacts Saved")

print("✓ processed_resume.csv")
print("✓ processed_jobs.csv")
print("✓ resume_embeddings.npy")
print("✓ job_embeddings_part_*.npy")
print("✓ job_faiss.index")
print()

print(f"Total Resumes : {len(resume_df)}")
print(f"Total Jobs    : {len(job_df2)}")
print("Embedding Dimension :", dimension)
print(f"FAISS Index Size    : {index.ntotal}")

print("=" * 70)

Notebook 1 Completed Successfully

Artifacts Saved
✓ processed_resume.csv
✓ processed_jobs.csv
✓ resume_embeddings.npy
✓ job_embeddings_part_*.npy
✓ job_faiss.index

Total Resumes : 2484
Total Jobs    : 1615940
Embedding Dimension : 384
FAISS Index Size    : 1615940


In [ ]:
import os

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

In [ ]:
resume_df.to_csv(DATA_PATH + "processed_resume.csv", index=False)
job_df2.to_csv(DATA_PATH + "processed_jobs.csv", index=False)

np.save(MODEL_PATH + "resume_embeddings.npy", resume_embeddings)
with open(MODEL_PATH + "embedding_model.txt", "w") as f:
    f.write(MODEL_NAME)

In [ ]:
DATA_PATH = "/content/drive/MyDrive/AI Resume Screening/"
MODEL_PATH = DATA_PATH + "models/"
OUTPUT_PATH = DATA_PATH + "output/"

In [ ]:
import sentence_transformers
import transformers
import PIL

print(sentence_transformers.__version__)
print(transformers.__version__)
print(PIL.__version__)

5.6.0
5.13.1
11.3.0


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Loaded successfully")

In [ ]:
import transformers

print(transformers.__version__)
print(transformers.__file__)